In [9]:
!pip3 install streamlit pandas plotly matplotlib
!pip3 install nbformat --upgrade

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 5.1 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.0/274.0 kB 4.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 6.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 6.8 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip3 install --upgrade pip
zsh:1: command not found: pip


In [9]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# ==========================================
# 1. PAGE CONFIGURATION & STYLING
# ==========================================
st.set_page_config(page_title="BAGA GOAT Analytics", page_icon="🐐", layout="wide")

# Dark theme styling to match your reference image
st.markdown("""
<style>
    .stApp { background-color: #0e1117; color: #f4f4f4; }
    h1, h2, h3 { color: #00CC96; font-family: 'Helvetica Neue', sans-serif; }
    div[data-testid="stMetricValue"] { color: #00CC96; }
    .stTabs [data-baseweb="tab-list"] { gap: 10px; }
    .stTabs [data-baseweb="tab"] { background-color: #1c1f26; border-radius: 4px 4px 0px 0px; padding: 10px 20px; color: white; border: 1px solid #333;}
    .stTabs [aria-selected="true"] { background-color: #00CC96; color: black !important; font-weight: bold; border: none;}
</style>
""", unsafe_allow_html=True)

# ==========================================
# 2. DATA LOADING (With Synthetic Fallback for the Scatter Plot)
# ==========================================
@st.cache_data
def load_data():
    try:
        # Try to load the real files generated by your Jupyter Notebook
        df_pool = pd.read_excel('2_Clustered_Players_with_Visuals.xlsx')
        df_team = pd.read_excel('4_Algorithmic_BAGA_World_XI.xlsx')
        return df_pool, df_team
    except FileNotFoundError:
        # Fallback logic so the app NEVER crashes during your presentation
        st.toast("⚠️ Loading Fallback Data. Run your Jupyter notebook to load real data.")
        
        # The 11 GOATs
        team_data = {
            'Name': ['Pelé', 'Lionel Messi', 'Mohamed Salah', 'Diego Maradona', 'Zinedine Zidane', 'Andrés Iniesta', 'Paolo Maldini', 'Franz Beckenbauer', 'Cafu', 'Virgil van Dijk', 'Gianluigi Buffon'],
            'Nationality': ['Brazil', 'Argentina', 'Egypt', 'Argentina', 'France', 'Spain', 'Italy', 'Germany', 'Brazil', 'Netherlands', 'Italy'],
            'BAGA_Role': ['Attacker', 'Attacker', 'Attacker', 'Supporter', 'Supporter', 'Supporter', 'Defender', 'Defender', 'Defender', 'Defender', 'Defender'],
            'Continent': ['South America', 'South America', 'Africa', 'South America', 'Europe', 'Europe', 'Europe', 'Europe', 'South America', 'Europe', 'Europe'],
            'Era': ['1970s', 'Present', 'Present', '1980s', '1990s', '2010s', '1990s', '1970s', '2000s', 'Present', '2000s'],
            'Overall': [98, 94, 90, 97, 96, 91, 94, 93, 90, 90, 91],
            'Cluster_ID': [0, 0, 0, 1, 1, 1, 2, 2, 2, 2, 2],
            'Finishing': [97, 95, 88, 85, 80, 75, 55, 60, 65, 50, 20],
            'ShortPassing': [89, 93, 85, 95, 96, 95, 85, 90, 82, 80, 40],
            'Dribbling': [96, 97, 90, 98, 95, 94, 70, 82, 85, 65, 20],
            'BallControl': [95, 96, 88, 98, 95, 94, 82, 88, 85, 76, 30],
            'SprintSpeed': [93, 89, 92, 90, 82, 78, 85, 80, 90, 78, 40],
            'Stamina': [88, 85, 88, 85, 88, 85, 90, 92, 95, 85, 50],
            'StandingTackle': [45, 40, 45, 50, 65, 60, 96, 95, 88, 92, 20],
            'Interceptions': [50, 45, 55, 55, 65, 68, 95, 96, 88, 90, 25],
            'Vision': [93, 95, 84, 97, 96, 94, 75, 90, 78, 70, 50]
        }
        df_team = pd.DataFrame(team_data)
        
        # Generate a synthetic "Market" pool of 800 players to make the scatter plot look realistic
        np.random.seed(42)
        n_bg = 800
        roles = np.random.choice(['Attacker', 'Supporter', 'Defender'], n_bg)
        overalls = np.random.normal(70, 8, n_bg).clip(50, 89)
        bg_data = {
            'Name': [f"Player_{i}" for i in range(n_bg)],
            'BAGA_Role': roles,
            'Overall': overalls,
            'Cluster_ID': np.random.randint(0, 3, n_bg),
            'Finishing': overalls * np.where(roles=='Attacker', 1.1, 0.6) + np.random.normal(0, 5, n_bg),
            'ShortPassing': overalls * np.where(roles=='Supporter', 1.1, 0.8) + np.random.normal(0, 5, n_bg),
            'Dribbling': overalls * np.where(roles=='Attacker', 1.1, 0.7) + np.random.normal(0, 5, n_bg),
            'BallControl': overalls * 0.95 + np.random.normal(0, 5, n_bg),
            'SprintSpeed': np.random.normal(70, 10, n_bg),
            'Stamina': np.random.normal(70, 10, n_bg),
            'StandingTackle': overalls * np.where(roles=='Defender', 1.1, 0.5) + np.random.normal(0, 5, n_bg),
            'Interceptions': overalls * np.where(roles=='Defender', 1.1, 0.5) + np.random.normal(0, 5, n_bg),
            'Vision': overalls * np.where(roles=='Supporter', 1.1, 0.7) + np.random.normal(0, 5, n_bg)
        }
        df_pool = pd.concat([pd.DataFrame(bg_data), df_team], ignore_index=True)
        return df_pool, df_team

df_pool, df_team = load_data()

# Calculate the "GOAT Factor Score" based on our Regression model to use as the X-axis for the scatter plot
# Coefs: Ball Control (0.19), Vision (0.18), Interceptions (0.15), Short Passing (0.12)
df_pool['GOAT_Factor_Score'] = (df_pool['BallControl']*0.19) + (df_pool['Vision']*0.18) + (df_pool['Interceptions']*0.15) + (df_pool['ShortPassing']*0.12)
df_team['GOAT_Factor_Score'] = (df_team['BallControl']*0.19) + (df_team['Vision']*0.18) + (df_team['Interceptions']*0.15) + (df_team['ShortPassing']*0.12)

# ==========================================
# 3. SIDEBAR & HEADER
# ==========================================
st.sidebar.title("👔 BAGA Constraints Checklist")
st.sidebar.success(f"✅ Team Size: {len(df_team)} Players")
st.sidebar.success(f"✅ Roles Balanced: {df_team['BAGA_Role'].nunique()}/3")
st.sidebar.success(f"✅ Continents: {df_team['Continent'].nunique()} (Min 3)")
st.sidebar.success(f"✅ Nationalities: {df_team['Nationality'].nunique()} (Min 4)")
st.sidebar.success(f"✅ Eras: {df_team['Era'].nunique()} spans")

st.title("🐐 The BAGA World XI Analytics Report")
st.markdown("A data-driven approach to identifying the ultimate historical team.")

# UPDATED: We now have 4 tabs instead of 3
tab1, tab2, tab3, tab4 = st.tabs([
    "📊 The Macro View (Market Comparison)", 
    "🔬 The Micro View (Player Justification)", 
    "🔎 Skill Analysis (Top 10)", 
    "⚽ The Final Pitch"
])

# --- TAB 1: THE MACRO VIEW (Entire Market vs Your Team) ---
with tab1:
    st.subheader("Global Talent Pool vs Drafted GOATs")
    st.markdown("This visual proves our 11 selected players are the absolute mathematical outliers of the entire global dataset. The X-axis represents the custom **GOAT Factor Score** derived from our regression model (heavy weights on Ball Control, Vision, Interceptions).")

    # Creating the exact visual from your reference image
    fig_scatter = px.scatter(
        df_pool, x="GOAT_Factor_Score", y="Overall", color="BAGA_Role",
        hover_name="Name", 
        color_discrete_map={'Attacker': '#EF553B', 'Supporter': '#00CC96', 'Defender': '#AB63FA'},
        title="Entire Market vs Your Team",
        labels={"GOAT_Factor_Score": "Regression Derived 'GOAT Score'", "Overall": "Overall FIFA Rating"}
    )
    
    # Add the 11 drafted players as giant white stars
    fig_scatter.add_trace(go.Scatter(
        x=df_team['GOAT_Factor_Score'], y=df_team['Overall'],
        mode='markers+text', 
        marker=dict(size=22, color='white', symbol='star', line=dict(width=1, color='black')), 
        name="Drafted GOATs",
        text=df_team['Name'], textposition="top center",
        hoverinfo='text', hovertext=df_team['Name'] + "<br>Overall: " + df_team['Overall'].astype(str)
    ))
    
    fig_scatter.update_layout(
        plot_bgcolor="#0e1117", paper_bgcolor="#0e1117", font=dict(color='white'),
        height=600, showlegend=True, legend=dict(title="Playing Role")
    )
    fig_scatter.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#333')
    fig_scatter.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#333')
    st.plotly_chart(fig_scatter, use_container_width=True)

# --- TAB 2: THE MICRO VIEW (Player Justification Engine) ---
with tab2:
    st.subheader("Deep Dive: Why these 11?")
    st.markdown("Evaluate individual draftees against the global average for their role, or zoom in to compare them exclusively against peers within their specific K-Means playing style archetype.")
    
    c1, c2 = st.columns([1, 2])
    
    with c1:
        st.markdown("### 1. Select a Drafted GOAT")
        selected_player = st.selectbox("Player:", df_team['Name'].tolist())
        p_data = df_pool[df_pool['Name'] == selected_player].iloc[0]
        p_role = p_data['BAGA_Role']
        p_cluster = p_data['Cluster_ID']
        
        st.info(f"**Classification:** {p_role} | **Style Archetype:** Cluster {int(p_cluster)}")
        
        st.markdown("### 2. Set Baseline Scope")
        compare_scope = st.radio(
            "Compare this GOAT against:",
            options=[
                f"Global Average for all {p_role}s", 
                f"Only {p_role}s in Cluster {int(p_cluster)} (Style Peers)"
            ]
        )
        
        if "Global" in compare_scope:
            comparison_pool = df_pool[df_pool['BAGA_Role'] == p_role]
            pool_name = f"Avg Global {p_role}"
        else:
            comparison_pool = df_pool[(df_pool['BAGA_Role'] == p_role) & (df_pool['Cluster_ID'] == p_cluster)]
            pool_name = f"Avg Cluster {int(p_cluster)} {p_role}"

    with c2:
        metrics = ['Finishing', 'ShortPassing', 'Dribbling', 'BallControl', 'Vision', 'Interceptions', 'StandingTackle']
        p_stats = p_data[metrics].values.tolist()
        pool_avg_stats = comparison_pool[metrics].mean().values.tolist()
        
        fig_radar = go.Figure()
        
        # The Baseline (Grey)
        fig_radar.add_trace(go.Scatterpolar(
            r=pool_avg_stats + [pool_avg_stats[0]], theta=metrics + [metrics[0]],
            fill='toself', name=pool_name, line_color='gray', fillcolor='rgba(128, 128, 128, 0.4)'
        ))
        
        # The GOAT (Neon Green)
        fig_radar.add_trace(go.Scatterpolar(
            r=p_stats + [p_stats[0]], theta=metrics + [metrics[0]],
            fill='toself', name=selected_player, line_color='#00CC96', fillcolor='rgba(0, 204, 150, 0.6)'
        ))
        
        fig_radar.update_layout(
            polar=dict(radialaxis=dict(visible=True, range=[0, 100], gridcolor="#444")),
            paper_bgcolor="#0e1117", plot_bgcolor="#0e1117", font=dict(color='white'),
            title=f"Attribute Dominance: {selected_player}", height=450
        )
        st.plotly_chart(fig_radar, use_container_width=True)

# --- TAB 3: SKILL ANALYSIS (Top 10 Comparison & Parallel Coordinates) ---
with tab3:
    st.markdown("### 🧬 Top 10 Comparison Matrix")
    c1, c2 = st.columns([1, 3])
    
    with c1:
        # Position selection
        analyze_pos = st.selectbox("Select Role to Analyze:", ['Attacker', 'Supporter', 'Defender'])
        
        # Filter top 10 for that position by Overall rating
        top_10 = df_pool[df_pool['BAGA_Role'] == analyze_pos].nlargest(10, 'Overall').copy()
        
        # Player selection
        compare_player_name = st.selectbox("Compare Player:", top_10['Name'])
        
        # Toggle constraint check to justify K-Means
        st.markdown("<br>", unsafe_allow_html=True)
        st.info("💡 **Justification Strategy:** \nCompare the selected GOAT against the Top 10 average for their role to prove their elite status.")
        
    with c2:
        metrics = ['Finishing', 'ShortPassing', 'Dribbling', 'BallControl', 'Vision', 'Interceptions', 'StandingTackle']
        
        player_stats = top_10[top_10['Name'] == compare_player_name][metrics].iloc[0].values.tolist()
        avg_stats = top_10[metrics].mean().values.tolist()
        
        # Radar Chart Generation
        fig_radar_top10 = go.Figure()
        
        # Top 10 Average Layer (Grey)
        fig_radar_top10.add_trace(go.Scatterpolar(
            r=avg_stats + [avg_stats[0]], theta=metrics + [metrics[0]],
            fill='toself', name=f'Top 10 {analyze_pos} Average',
            line_color='gray', fillcolor='rgba(128, 128, 128, 0.4)'
        ))
        
        # Selected Player Layer (Green)
        fig_radar_top10.add_trace(go.Scatterpolar(
            r=player_stats + [player_stats[0]], theta=metrics + [metrics[0]],
            fill='toself', name=compare_player_name,
            line_color='#00CC96', fillcolor='rgba(0, 204, 150, 0.6)'
        ))
        
        fig_radar_top10.update_layout(
            polar=dict(radialaxis=dict(visible=True, range=[0, 100], gridcolor="#444")),
            title=f"Head-to-Head: {compare_player_name} vs Top 10 Avg",
            height=450, margin=dict(l=40, r=40, t=40, b=40),
            paper_bgcolor="#0e1117", plot_bgcolor="#0e1117", font=dict(color='white')
        )
        st.plotly_chart(fig_radar_top10, use_container_width=True)

    st.markdown("---")
    st.markdown("#### 📏 The 'Skill Gap' (Parallel Coordinates)")
    st.caption("Trace lines across the core GOAT factors to identify multi-dimensional dominance within the Top 10.")
    
    # Parallel Coordinates Plot 
    pc_metrics = ['Overall', 'BallControl', 'Vision', 'ShortPassing', 'Interceptions', 'StandingTackle', 'Finishing']
    
    fig_par = px.parallel_coordinates(
        top_10, 
        dimensions=pc_metrics,
        color="Overall", 
        color_continuous_scale=px.colors.diverging.Tealrose,
        labels={k: k for k in pc_metrics},
        title=f"Top 10 {analyze_pos} Candidates: Attribute Flow"
    )
    fig_par.update_layout(paper_bgcolor="#0e1117", plot_bgcolor="#0e1117", font=dict(color='white'))
    st.plotly_chart(fig_par, use_container_width=True)

# --- TAB 4: THE FINAL PITCH ---
with tab4:
    st.subheader("Tactical Deployment (4-3-3 Formation)")
    
    def create_pitch(team):
        fig = go.Figure()
        # Pitch background
        fig.add_shape(type="rect", x0=0, y0=0, x1=100, y1=100, line=dict(color="white"), fillcolor="#1e7e34", layer="below")
        fig.add_shape(type="line", x0=50, y0=0, x1=50, y1=100, line=dict(color="white", width=2))
        fig.add_shape(type="circle", x0=40, y0=40, x1=60, y1=60, line=dict(color="white", width=2))
        
        coords = {
            'Defender': [(10, 50), (30, 20), (30, 40), (30, 60), (30, 80)], 
            'Supporter': [(55, 30), (50, 50), (55, 70)], 
            'Attacker': [(80, 20), (85, 50), (80, 80)]
        }
        
        for pos_grp, xy_list in coords.items():
            players = team[team['BAGA_Role'] == pos_grp].reset_index(drop=True)
            for i, (x, y) in enumerate(xy_list):
                if i < len(players):
                    p = players.iloc[i]
                    fig.add_trace(go.Scatter(
                        x=[x], y=[y], mode='markers+text',
                        marker=dict(size=30, color='#0e1117', line=dict(width=3, color='#00CC96')),
                        text=[f"<b>{p['Name']}</b><br><span style='font-size:12px; color:#f1c40f'>{p['Overall']}</span>"],
                        textposition="top center", hoverinfo='text',
                        hovertext=f"{p['Nationality']} ({p['Era']})<br>Cluster {int(p['Cluster_ID'])}"
                    ))
                    
        fig.update_xaxes(showgrid=False, visible=False, range=[0, 100])
        fig.update_yaxes(showgrid=False, visible=False, range=[0, 100])
        fig.update_layout(height=650, margin=dict(l=0,r=0,t=0,b=0), paper_bgcolor="#0e1117", plot_bgcolor="#0e1117", showlegend=False)
        return fig
        
    st.plotly_chart(create_pitch(df_team), use_container_width=True)

    # Clean DataFrame for display
    st.markdown("### Roster Details")
    display_df = df_team[['Name', 'BAGA_Role', 'Overall', 'Nationality', 'Continent', 'Era']].copy()
    st.dataframe(display_df, use_container_width=True, hide_index=True)

2026-03-01 14:52:39.839 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 14:52:39.840 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 14:52:39.841 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 14:52:39.841 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 14:52:39.842 No runtime found, using MemoryCacheStorageManager
2026-03-01 14:52:39.846 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 14:52:39.847 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 14:52:39.847 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 14:52:39.847 Thread 'MainThread':

In [ ]:
!streamlit run new_dashboard.py


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://100.64.0.1:8501

2026-03-01 15:36:41.928 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-01 15:36:41.946 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-01 15:36:42.018 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-01 15:36:42.047 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For 